In [1]:
import os
import sys
notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, '..')) # Adjust as needed
if project_root not in sys.path:
    sys.path.append(project_root) # add notebook to sys.path

In [2]:
import torch
from torch import nn
import torch.nn.functional as F

In [3]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"torch.accelerater.current_accelerator() gives {device} device")

torch.accelerater.current_accelerator() gives mps device


In [4]:
class LeNetCNN(nn.Module):
    def __init__(self, num_channels=16, dropout=0.5):
        super().__init__()
        self.num_channels = num_channels
        self.conv1 = nn.LazyConv2d(6, kernel_size=5, padding=2)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.LazyConv2d(16, kernel_size=5, padding=2)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.linear1 = nn.LazyLinear(120)
        self.linear2 = nn.LazyLinear(84)
        self.linear_out = nn.LazyLinear(10)
        self.net = nn.Sequential(
            self.conv1, nn.ReLU(), self.pool1,
            self.conv2, nn.ReLU(), self.pool2,
            nn.Flatten(), 
            self.linear1, nn.ReLU(), nn.Dropout(dropout),
            self.linear2, nn.ReLU(), nn.Dropout(dropout),
            self.linear_out
        )
    
    def forward(self, X):
        return self.net(X)

# First dataset: MNIST

In [5]:
from utils.data import MNIST

In [6]:
batch_size = 64
mnist = MNIST(batch_size, resize=(32, 32), device=device)
train_dl, val_dl = mnist.get_dataloaders()

In [21]:
model = LeNetCNN().to(device)
# init lazy layers
X, y = next(iter(train_dl))
print(X.shape, y.shape)
print(X.device, y.device)
print(model(X).shape)

torch.Size([64, 3, 64, 64]) torch.Size([64])
mps:0 mps:0
torch.Size([64, 10])


In [8]:
opt = torch.optim.Adam(model.parameters(), lr=0.001)

In [9]:
# one epoch of training

losses = []

for X, y in train_dl:
    logits = model(X)
    loss = F.cross_entropy(logits, y)
    loss.backward()
    opt.step()
    opt.zero_grad()
    losses.append(loss.item())

In [10]:
print(losses[0], losses[-1])

2.3131794929504395 0.24080437421798706


# Second dataset: FashionMNIST

In [11]:
from utils.data import FashionMNIST

fashion_mnist = FashionMNIST(batch_size, resize=(32, 32), device=device)
train_dl, val_dl = fashion_mnist.get_dataloaders()

In [20]:
model = LeNetCNN().to(device)
# init lazy layers
X, y = next(iter(train_dl))
print(X.shape, y.shape)
print(X.device, y.device)
print(model(X).shape)
opt = torch.optim.Adam(model.parameters(), lr=0.001)

torch.Size([64, 3, 64, 64]) torch.Size([64])
mps:0 mps:0
torch.Size([64, 10])


In [13]:
# one epoch of training

losses = []

for X, y in train_dl:
    logits = model(X)
    loss = F.cross_entropy(logits, y)
    loss.backward()
    opt.step()
    opt.zero_grad()
    losses.append(loss.item())

In [14]:
print(losses[0], losses[-1])

2.301867961883545 0.689130425453186


# Third dataset: CIFAR-10

In [15]:
from utils.data import CIFAR10

fashion_mnist = CIFAR10(batch_size, resize=(64, 64), device=device)
train_dl, val_dl = fashion_mnist.get_dataloaders()

In [19]:
model = LeNetCNN().to(device)
# init lazy layers
X, y = next(iter(train_dl))
print(X.shape, y.shape)
print(X.device, y.device)
print(model(X).shape)
opt = torch.optim.Adam(model.parameters(), lr=0.001)

torch.Size([64, 3, 64, 64]) torch.Size([64])
mps:0 mps:0
torch.Size([64, 10])


In [17]:
# one epoch of training

losses = []

for X, y in train_dl:
    logits = model(X)
    loss = F.cross_entropy(logits, y)
    loss.backward()
    opt.step()
    opt.zero_grad()
    losses.append(loss.item())

In [18]:
print(losses[0], losses[-1])

2.303624391555786 1.5308881998062134
